# GitHub Events ETL Pipeline

An end-to-end pipeline that **Extracts** GitHub repository events via the REST API,  
**Transforms** the nested JSON into a flat structure, and **Loads** the result into a DuckDB database.

**Pipeline stages:**
1. Environment Setup
2. Extraction — fetch events with rate-limit handling & pagination
3. Transformation — flatten nested fields, parse timestamps
4. Loading — upsert into DuckDB with schema evolution support

---
## 1. Environment Setup

In [ ]:
# Install required libraries (uncomment when running for the first time)
# %pip install requests duckdb

In [ ]:
import os
import time
import requests
import duckdb
from datetime import datetime

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
REPO_URL = "https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp/events"
RATE_LIMIT_URL = "https://api.github.com/rate_limit"
DB_PATH = "github_events.db"
TABLE_NAME = "github_events"

# Auth token raises the rate limit from 60 → 5,000 requests/hour
# Set via environment variable — never hardcode tokens in notebooks
API_TOKEN = os.getenv("GITHUB_TOKEN")
HEADERS = {"Authorization": f"Bearer {API_TOKEN}"} if API_TOKEN else {}

---
## 2. Extraction

The GitHub Events API returns paginated results (up to 30 events/page).  
We use the `Link` response header to follow pages until no `next` link exists.

A **generator** is used instead of collecting all pages in a list,  
keeping memory consumption constant regardless of the total number of events.

In [ ]:
def check_rate_limit() -> int:
    """Return the number of remaining API requests. Sleeps 60 s if exhausted."""
    remaining = requests.get(RATE_LIMIT_URL, headers=HEADERS).json()["rate"]["remaining"]
    if remaining == 0:
        print("Rate limit reached — waiting 60 seconds...")
        time.sleep(60)
    return remaining

print(f"Remaining API requests: {check_rate_limit()}")

In [ ]:
def fetch_events(url: str = REPO_URL):
    """
    Generator — yields one page of GitHub events at a time.

    Follows the `rel='next'` Link header to traverse all pages.
    Each page is a list of raw event dicts from the GitHub API.
    """
    while True:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        yield response.json()

        if "next" not in response.links:
            break  # No more pages
        url = response.links["next"]["url"]

In [ ]:
# Fetch all pages and collect raw events
all_events = []

for page in fetch_events():
    all_events.extend(page)

print(f"Total events fetched: {len(all_events)}")

---
## 3. Transformation

Each raw event is a deeply nested dict. We flatten the key fields into a single-level record:
- Actor fields are prefixed with `actor__`
- Repo fields are prefixed with `repo__`
- `created_at` (ISO string) is converted to a Unix timestamp for numeric sorting

In [ ]:
def process_event(event: dict) -> dict | None:
    """
    Flatten a raw GitHub event dict into a clean, single-level record.

    Returns None for malformed entries (e.g., API error responses).
    """
    if not isinstance(event, dict) or "id" not in event:
        return None

    return {
        "id":           event["id"],
        "type":         event["type"],
        "public":       event["public"],
        "created_at":   datetime.fromisoformat(event["created_at"]).timestamp(),
        "actor__id":    event["actor"]["id"],
        "actor__login": event["actor"]["login"],
        "repo__id":     event["repo"]["id"],
    }

In [ ]:
processed_events = []
skipped = 0

for event in all_events:
    record = process_event(event)
    if record:
        processed_events.append(record)
    else:
        skipped += 1

print(f"Processed: {len(processed_events)} events | Skipped: {skipped} malformed entries")
print("Sample record:", processed_events[0] if processed_events else "(none)")

---
## 4. Loading — DuckDB

We use **DuckDB** as a lightweight local analytical database.

Key design choices:
- `id` is the primary key — duplicate events are safely ignored (`ON CONFLICT DO NOTHING`)
- Schema evolution: new columns found in incoming data are added automatically with `ALTER TABLE`
- Records are inserted in bulk with `executemany` for performance

In [ ]:
conn = duckdb.connect(DB_PATH)
print(f"Connected to DuckDB: {DB_PATH}")

In [ ]:
# Create the target table if it does not already exist
conn.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id           TEXT PRIMARY KEY,
    type         TEXT,
    public       BOOLEAN,
    created_at   DOUBLE,
    actor__id    BIGINT,
    actor__login TEXT,
    repo__id     BIGINT
);
""")
print(f"Table `{TABLE_NAME}` is ready.")

In [ ]:
# Schema evolution — add any new columns introduced by the incoming data
existing_cols = {
    row[1] for row in conn.execute(f"PRAGMA table_info({TABLE_NAME})").fetchall()
}

TYPE_MAP = {bool: "BOOLEAN", int: "BIGINT", float: "DOUBLE"}

for record in processed_events:
    for col, val in record.items():
        if col not in existing_cols:
            col_type = TYPE_MAP.get(type(val), "TEXT")
            conn.execute(f"ALTER TABLE {TABLE_NAME} ADD COLUMN {col} {col_type};")
            existing_cols.add(col)
            print(f"  Added column: {col} ({col_type})")

print("Schema check complete.")

In [ ]:
# Build and execute a bulk upsert (insert, skip duplicates)
columns = sorted(existing_cols)
placeholders = ", ".join(["?" for _ in columns])
columns_str = ", ".join(columns)

insert_sql = f"""
INSERT INTO {TABLE_NAME} ({columns_str})
VALUES ({placeholders})
ON CONFLICT (id) DO NOTHING;
"""

# Flatten each record into an ordered tuple matching the column list
rows = [tuple(rec.get(col) for col in columns) for rec in processed_events]

conn.executemany(insert_sql, rows)
print(f"Inserted {len(rows)} rows into `{TABLE_NAME}`.")

---
## 5. Verification

In [ ]:
# Quick sanity check — preview the loaded data
df = conn.execute(f"SELECT * FROM {TABLE_NAME} ORDER BY created_at DESC LIMIT 5").df()
print(f"Total rows in DB: {conn.execute(f'SELECT COUNT(*) FROM {TABLE_NAME}').fetchone()[0]}")
df